In [ ]:
import os

from IPython.display import display
import downstream
import imageio
import marimo as mo
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm
from watermark import watermark

from pylib._osf_fetch import osf_fetch
from pylib.munge._addend_groups import addend_groups
from pylib.munge._squeeze import squeeze_int
from pylib.munge._union_upsample import union_upsample
from pylib.theory._calc_denovo_peak_prevalence_demes2d import (
    calc_denovo_peak_prevalence_demes2d,
)

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-25T22:41:37.426278+00:00

Python implementation: CPython
Python version       : 3.10.22
IPython version      : 7.31.1

Compiler    : GCC 13.3.0
OS          : Linux
Release     : 7.0.0-1018-aws
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

teeplot   : 1.2.0
imageio   : 2.36.1
pandas    : 2.2.3
polars    : 1.9.0
seaborn   : 0.13.2
downstream: 1.6.2
marimo    : 0.23.2
numpy     : 2.1.2

```

In [ ]:
teeplot_subdir = (
    "wse-denovo-spatial2d-explicitsite-timeseries-2025-02-07-peaksweep"
)
teeplot_subdir

'wse-denovo-spatial2d-explicitsite-timeseries-2025-02-07-peaksweep'

## Prep Data

In [ ]:
dfxs = pl.from_pandas(
    pd.read_parquet(osf_fetch("https://osf.io/yaedh/download")),
).with_row_index(
    "dstream_data_id",
)
display(dfxs.describe()), display(dfxs.head()), display(dfxs.tail())

downloading https://osf.io/yaedh/download -> /tmp/yaedh
size: 61624790 bytes


In [ ]:
def unpack_logger_timeseries_data(df: pl.DataFrame) -> pl.DataFrame:

    # sample every 64th tile
    df = df.filter((pl.col("col") % 8 == 0) & (pl.col("row") % 8 == 0))

    # unpack dstream data items from logger hex strings
    dfu = downstream.dataframe.unpack_data_packed(df)
    df = downstream.dataframe.explode_lookup_unpacked(
        dfu.with_columns(
            # fast forward loggers to at least dstream_S entries
            # (i.e., buffer filled)
            # these will be semantically-correct zero values
            dstream_T=pl.max_horizontal(
                pl.col("dstream_T"), pl.col("dstream_S")
            ),
        ),
        value_type="uint8",
    )

    # join metadata onto unpacked data
    df = df.join(
        dfu.select(
            "dstream_data_id",
            "CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS",
            "CEREBRASLIB_TRAITLOGGER_DILATION",
            "col",
            "replicate",
            "row",
            "tile",
        ),
        on="dstream_data_id",
    )

    # apply dilation to dstream_T and dstream_Tbar
    df = df.with_columns(
        dstream_T=pl.col("dstream_T")
        * pl.col("CEREBRASLIB_TRAITLOGGER_DILATION"),
        dstream_Tbar=pl.col("dstream_Tbar")
        * pl.col("CEREBRASLIB_TRAITLOGGER_DILATION"),
    )

    # addend time time series groups with final zero value entry
    df = addend_groups(
        df,
        group_by=["replicate", "tile"],
        aggs=(
            pl.col("CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS")
            .unique()
            .pipe(squeeze_int),
            pl.col("col").unique().pipe(squeeze_int),
            (pl.col("dstream_Tbar") + 1).max().alias("dstream_Tbar"),
            pl.lit(0).alias("dstream_value"),
            pl.col("row").unique().pipe(squeeze_int),
        ),
        inner_only=True,
    )

    # upsample to fill in dstream_Tbar dropped by some loggers
    # prevents ampling biases from longer-running vs shorter-running loggers
    df = union_upsample(
        df,
        upsample="dstream_Tbar",
        group_by="tile",
        fill_null_ops=[
            (pl.all(), {"strategy": "forward"}),
            (pl.col("dstream_Tbar", "dstream_value"), {"strategy": "zero"}),
            (pl.all(), {"strategy": "backward"}),
        ],
    )
    assert len(df["dstream_Tbar"].value_counts()["count"].unique()) == 1

    return df

In [ ]:
res = []
for _, df in tqdm(dfxs.group_by("replicate")):
    df = unpack_logger_timeseries_data(df)
    res.append(df)

df = pl.concat(res)

0it [00:00, ?it/s]/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/downstream/dataframe/_impl/_check_downstream_version.py:29: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
1it [00:01,  1.70s/it]/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/downstream/dataframe/_impl/_check_downstream_version.py:29: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
2it [00:03,  1.60s/it]/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/downstream/dataframe/_impl/_check_downstream_version.py:29: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
3it [00:04,  1.60s/it]/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/downstream/dataframe/_impl/_check_downstream_version.py:29: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
4it [00:06,  1.64s/it]/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-pa

## Plot Data

In [ ]:
# take the mean of dstream_value at each dstream_Tbar across tiles
df_mean = df.group_by("replicate", "dstream_Tbar").agg(
    pl.col("dstream_value").mean(),
    pl.col("CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS")
    .unique()
    .pipe(squeeze_int),
)

In [ ]:
with tp.teed(
    sns.relplot,
    data=df_mean.with_columns(
        pl.col("replicate")
        .rank(method="dense")
        .over("CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS")
        .cast(pl.Int32)
        .alias("replicate number")
    ),
    x="dstream_Tbar",
    y="dstream_value",
    hue="replicate",
    col="replicate number",
    row="CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS",
    alpha=0.5,
    aspect=1.5,
    errorbar=None,
    height=1.2,
    linewidth=1,
    legend=False,
    kind="line",
    marker="d",
    markersize=4,
    markers=True,
    facet_kws=dict(margin_titles=True),
    teeplot_subdir=teeplot_subdir,
    teeplot_outexclude=["marker"],
) as _g:
    _g.set(xlim=(10, 80000))
    _g.set(ylim=(0.00001, 2))
    _g.set_titles(
        col_template="",
        row_template="{row_name} Avail\nBen Muts",
    )
    _g.set_axis_labels(x_var="Generations", y_var="Normomut\nPrevalence")
    for _ax in _g.axes.flat:
        _ax.axhline(1, ls="--", color="black", linewidth=1, zorder=-10)
        _ax.set_yscale("log")
        _ax.set_xscale("log")

/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: Tight layout not applied. tight_layout cannot make Axes height small enough to accommodate all Axes decorations.
  self._figure.tight_layout(*args, **kwargs)
/opt/hostedtoolcache/Python/3.10.22/x64/lib/python3.10/site-packages/seaborn/axisgrid.py:123: UserWarning: Tight layout not applied. tight_layout cannot make Axes height small enough to accommodate all Axes decorations.
  self._figure.tight_layout(*args, **kwargs)


teeplots/wse-denovo-spatial2d-explicitsite-timeseries-2025-02-07-peaksweep/col=replicate-number+hue=replicate+kind=line+row=cerebraslib-hypermut-num-avail-ben-muts+viz=relplot+x=dstream-tbar+y=dstream-value+ext=.pdf
teeplots/wse-denovo-spatial2d-explicitsite-timeseries-2025-02-07-peaksweep/col=replicate-number+hue=replicate+kind=line+row=cerebraslib-hypermut-num-avail-ben-muts+viz=relplot+x=dstream-tbar+y=dstream-value+ext=.png


In [ ]:
# take the mean of dstream_value at each dstream_Tbar across tiles
df_min_mean = (
    df.group_by("replicate", "dstream_Tbar")
    .agg(
        pl.col("dstream_value").mean(),
        pl.col("CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS")
        .unique()
        .pipe(squeeze_int),
    )
    .group_by("replicate")
    .agg(
        pl.col("dstream_value").filter(pl.col("dstream_value") > 0).min(),
        pl.col("CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS")
        .unique()
        .pipe(squeeze_int),
    )
)
df_min_mean

replicate,dstream_value,CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS
cat,f64,i64
"""76e9aab2-0e99-4bcc-a14e-e4678d…",0.091319,8
"""2a68b3e6-521c-4ece-866c-6e169e…",0.96366,2
"""8718a221-f4ea-4f41-8839-d93500…",0.31966,6
"""dd970b05-40a0-4825-81f4-c52bfa…",0.004851,10
"""4269e405-b8ec-4d45-9621-73ea45…",0.308766,6
…,…,…
"""715c2c6e-0c13-4f8f-8357-93be59…",0.754383,4
"""e8f182bf-02f8-472e-a882-3e9ecc…",0.003234,10
"""25555b9d-eaab-4c0e-ab38-256eaf…",0.960766,2


In [ ]:
with tp.teed(
    sns.relplot,
    data=df_min_mean.with_columns(
        pl.col("replicate")
        .rank(method="dense")
        .over("CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS")
        .cast(pl.Int32)
        .alias("replicate number")
    ).with_columns(dstream_value=1 - pl.col("dstream_value")),
    x="CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS",
    y="dstream_value",
    alpha=0.5,
    aspect=1.8,
    color="orange",
    errorbar=("pi", 100),
    height=1.4,
    linewidth=1,
    legend=False,
    kind="line",
    marker="X",
    markersize=7,
    markers=True,
    facet_kws=dict(margin_titles=True),
    teeplot_subdir=teeplot_subdir,
    teeplot_outexclude=["marker"],
) as _g:
    # g.set(xlim=(10, 80000))
    # g.set(ylim=(0.00001, 2))
    _g.set_titles(
        col_template="",
        row_template="{row_name} Avail\nBen Muts",
    )
    _g.set_axis_labels(
        x_var="Avail Ben Muts", y_var="Max\nMutator\nPrevalence"
    )
    # math model prediction (see pylib/theory), for the 750 x 994 grid
    # of 256-agent demes simulated here
    _model_K = [*range(2, 11)]
    _g.ax.plot(
        _model_K,
        [
            calc_denovo_peak_prevalence_demes2d(750 * 994 * 256, K)
            for K in _model_K
        ],
        color="black",
        linestyle="--",
    )
    for _ax in _g.axes.flat:
        _ax.axhline(1, ls="--", color="black", linewidth=1, zorder=-20)
        _ax.axhline(0.5, ls=":", color="black", linewidth=1, zorder=-20)
        _ax.set_yticks([0, 0.5, 1])

teeplots/wse-denovo-spatial2d-explicitsite-timeseries-2025-02-07-peaksweep/color=orange+kind=line+viz=relplot+x=cerebraslib-hypermut-num-avail-ben-muts+y=dstream-value+ext=.pdf
teeplots/wse-denovo-spatial2d-explicitsite-timeseries-2025-02-07-peaksweep/color=orange+kind=line+viz=relplot+x=cerebraslib-hypermut-num-avail-ben-muts+y=dstream-value+ext=.png


## Animate Data

In [ ]:
os.makedirs(teeplot_subdir, exist_ok=True)

for (replicate, nmut), dfr in tqdm(
    df.group_by("replicate", "CEREBRASLIB_HYPERMUT_NUM_AVAIL_BEN_MUTS"),
):
    frames = []
    for (dstream_Tbar,), dff in dfr.sort("dstream_Tbar").group_by(
        "dstream_Tbar", maintain_order=True
    ):

        dff = dff.with_columns(
            col_rank=pl.col("col").rank(method="dense").cast(pl.Int32) - 1,
            row_rank=pl.col("row").rank(method="dense").cast(pl.Int32) - 1,
        )

        num_rows = dff["row_rank"].max() + 1
        num_cols = dff["col_rank"].max() + 1

        frame = np.empty((num_rows, num_cols), dtype=np.uint8)
        frame[dff["row_rank"], dff["col_rank"]] = (
            dff["dstream_value"] * 255
        )
        frame = np.kron(frame, np.ones((4, 4), dtype=frame.dtype))
        frames.append(frame)

        outdir = (
            f"{teeplot_subdir}/a=traitframes+nmut={nmut}+rep={replicate}"
        )
        os.makedirs(outdir, exist_ok=True)
        outpath = f"{outdir}/dstream_Tbar={dstream_Tbar:06}+ext=.png"
        imageio.imwrite(outpath, frame.T)

    print(len(frames))
    outpath = (
        f"{teeplot_subdir}/a=traits+nmut={nmut}+rep={replicate}+ext=.gif"
    )
    imageio.mimsave(outpath, frames, duration=0.2)

0it [00:00, ?it/s]

257


1it [00:04,  4.65s/it]

257


2it [00:06,  3.24s/it]

257


... (31 outputs left out)

257


19it [00:42,  2.15s/it]

257


20it [00:44,  2.22s/it]
